# Retrieval evaluation

This notebook evaluates the retrieval layer over the already-generated chunks.

Production design:

1. resolve fund names from the validated registry;
2. apply chunk-level fund filtering;
3. use **BGE-M3** for multilingual dense retrieval;
4. `auto` mode uses:
   - semantic retrieval for mixed English/Bulgarian candidate pools;
   - hybrid BGE + language-aware BM25 for monolingual pools;
5. optionally build a diversified top-50 candidate set and rerank it with a multilingual cross-encoder.

The previous controlled benchmark motivated the switch from `multilingual-e5-base` to BGE-M3:

- EN → BG transfer: rank **49 → 2**
- EN → BG rights: rank **31 → 4**
- EN → EN and BG → EN remained rank **1**
- BGE-M3 Recall@5 was **1.0** across the six multilingual benchmark cases.

No ingestion, fund discovery, segmentation, or chunking is rerun here.

In [ ]:
import os
import sys
import json
import gc
import importlib
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

os.environ.setdefault(
    "HF_HOME",
    str(ROOT / ".cache" / "huggingface"),
)
os.environ.setdefault(
    "HF_HUB_CACHE",
    str(ROOT / ".cache" / "huggingface" / "hub"),
)

sys.path.insert(0, str(ROOT / "src"))

from fund_rag import retrieval
importlib.reload(retrieval)

CHUNK_PATH = ROOT / "artifacts" / "chunks" / "chunks.json"
REGISTRY_PATH = ROOT / "artifacts" / "fund_discovery" / "registry.json"
INDEX_DIR = ROOT / "artifacts" / "retrieval_bge_m3"

chunks = json.loads(CHUNK_PATH.read_text(encoding="utf-8"))
registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))

print("Chunks:", len(chunks))
print("Registry funds:", len(registry))
print("Dense model:", retrieval.DEFAULT_MODEL)
print("HF cache:", os.environ["HF_HOME"])

## 1. Build/load the production retriever

The first BGE-M3 run builds the FAISS index. Later runs reuse it as long as the model, indexed representation, and chunks are unchanged.

In [ ]:
REBUILD_INDEX = False

retriever = retrieval.HybridRetriever(
    chunks=chunks,
    registry=registry,
    index_dir=INDEX_DIR,
    rebuild=REBUILD_INDEX,
    batch_size=4,
)

print("FAISS vectors:", retriever.index.ntotal)

## 2. Routing checks

In [ ]:
# No named fund -> shared/no-fund chunks only.
results = retriever.retrieve(
    "What rights do I have as an investor?",
    k=10,
    mode="semantic",
)
assert results
assert all(r.get("fund") is None for r in results)

# Named fund -> only that fund.
results = retriever.retrieve(
    "What is the investment objective of K&H egészség?",
    k=10,
    mode="semantic",
)
assert results
assert all(r.get("fund") == "K&H egészség" for r in results)

# Overlapping names must resolve to the longest known fund.
assert retrieval.detect_query_funds(
    "What is the objective of K&H egészség 2?",
    registry,
) == ["K&H egészség 2"]

print("Routing checks passed.")

## 3. Evidence-level evaluation cases

Broad questions can require evidence from multiple chunks.
For example, investor rights may be distributed across redemption, information, and complaint sections, so evaluation measures **coverage across the top-k retrieved set**, not whether one chunk contains every required fact.

In [ ]:
CASES = [
    {
        "name": "rights_transfer_en",
        "direction": "EN -> BG",
        "query": "Can I transfer my fund units to another person?",
        "expected_document": "11074.pdf",
        "evidence_groups": [
            ["прехвърляне на дялове", "могат да се прехвърлят свободно"],
        ],
    },
    {
        "name": "rights_transfer_bg",
        "direction": "BG -> BG",
        "query": "Мога ли да прехвърля дяловете си на друго лице?",
        "expected_document": "11074.pdf",
        "evidence_groups": [
            ["прехвърляне на дялове", "могат да се прехвърлят свободно"],
        ],
    },
    {
        "name": "rights_general_en",
        "direction": "EN -> BG",
        "query": "What rights do I have as an investor?",
        "expected_document": "11074.pdf",
        "evidence_groups": [
            ["право на обратно изкупуване"],
            ["право на информация"],
            ["право на жалба"],
        ],
    },
    {
        "name": "rights_general_bg",
        "direction": "BG -> BG",
        "query": "Какви права имам като инвеститор?",
        "expected_document": "11074.pdf",
        "evidence_groups": [
            ["право на обратно изкупуване"],
            ["право на информация"],
            ["право на жалба"],
        ],
    },
    {
        "name": "ubb_conservative_sustainable_en",
        "direction": "EN -> BG",
        "query": (
            "What is the minimum percentage of sustainable investments "
            "required for ОББ ЕкспертИйз Дефанзивен Консервативен "
            "Отговорно Инвестиращ?"
        ),
        "expected_fund": (
            "ОББ ЕкспертИйз Дефанзивен Консервативен "
            "Отговорно Инвестиращ"
        ),
        "evidence_groups": [
            ["устойчиви инвестиции"],
            ["поне 20%", "20,00 %", "20.00%"],
        ],
    },
    {
        "name": "ubb_conservative_sustainable_bg",
        "direction": "BG -> BG",
        "query": (
            "Какъв е минималният процент устойчиви инвестиции за "
            "ОББ ЕкспертИйз Дефанзивен Консервативен Отговорно Инвестиращ?"
        ),
        "expected_fund": (
            "ОББ ЕкспертИйз Дефанзивен Консервативен "
            "Отговорно Инвестиращ"
        ),
        "evidence_groups": [
            ["устойчиви инвестиции"],
            ["поне 20%", "20,00 %", "20.00%"],
        ],
    },
    {
        "name": "kh_health_object_en",
        "direction": "EN -> EN",
        "query": "What is the investment objective of K&H egészség?",
        "expected_fund": "K&H egészség",
        "evidence_groups": [
            [
                "investment objectives and strategy",
                "object of the sub-fund is twofold",
                "description of the sub-fund's object",
            ],
        ],
    },
    {
        "name": "kh_health_object_bg",
        "direction": "BG -> EN",
        "query": "Каква е инвестиционната цел на K&H egészség?",
        "expected_fund": "K&H egészség",
        "evidence_groups": [
            [
                "investment objectives and strategy",
                "object of the sub-fund is twofold",
                "description of the sub-fund's object",
            ],
        ],
    },
    {
        "name": "enhanced_esg_risk",
        "direction": "EN -> EN",
        "query": (
            "How does Enhanced Intelligence Global Allocation "
            "Responsible Investing evaluate ESG risk for companies?"
        ),
        "expected_fund": (
            "Enhanced Intelligence Global Allocation "
            "Responsible Investing"
        ),
        "evidence_groups": [
            ["esg risk score", "esg-(risk)score"],
            ["higher than 40", "score higher than 40"],
        ],
    },
]

In [ ]:
def source_matches(result, case):
    if (
        case.get("expected_document")
        and result.get("document") != case["expected_document"]
    ):
        return False

    if (
        case.get("expected_fund")
        and result.get("fund") != case["expected_fund"]
    ):
        return False

    return True


def result_text(result):
    return retrieval.norm(
        " ".join([
            result.get("section") or "",
            result.get("text") or "",
        ])
    )


def evaluate_case(results, case, k=5):
    top = results[:k]
    matching = [
        result
        for result in top
        if source_matches(result, case)
    ]

    combined = "\n".join(result_text(result) for result in matching)

    groups_found = [
        any(retrieval.norm(term) in combined for term in alternatives)
        for alternatives in case["evidence_groups"]
    ]

    first_answer_rank = None
    all_terms = [
        retrieval.norm(term)
        for alternatives in case["evidence_groups"]
        for term in alternatives
    ]

    for rank, result in enumerate(top, start=1):
        if not source_matches(result, case):
            continue
        text = result_text(result)
        if any(term in text for term in all_terms):
            first_answer_rank = rank
            break

    coverage = (
        sum(groups_found) / len(groups_found)
        if groups_found
        else 1.0
    )

    return {
        "source_hit": bool(matching),
        "evidence_coverage": coverage,
        "all_evidence": coverage == 1.0,
        "first_answer_rank": first_answer_rank,
        "rr": 1 / first_answer_rank if first_answer_rank else 0.0,
    }


def evaluate_mode(retriever_obj, mode):
    rows = []

    for case in CASES:
        results = retriever_obj.retrieve(
            case["query"],
            k=5,
            mode=mode,
        )
        metrics = evaluate_case(results, case, k=5)

        rows.append({
            "case": case["name"],
            "direction": case["direction"],
            "requested_mode": mode,
            "effective_mode": (
                results[0].get("effective_mode")
                if results
                else None
            ),
            **metrics,
        })

    return pd.DataFrame(rows)

## 4. Production policy: `auto`

`auto` deliberately avoids BM25 when the filtered candidate pool contains both English and Bulgarian chunks.
That keeps the strong cross-language BGE-M3 signal from being diluted by same-language lexical matches.

In [ ]:
auto_eval = evaluate_mode(retriever, "auto")
semantic_eval = evaluate_mode(retriever, "semantic")
hybrid_eval = evaluate_mode(retriever, "hybrid")

evaluation = pd.concat(
    [auto_eval, semantic_eval, hybrid_eval],
    ignore_index=True,
)

display(evaluation)

summary = (
    evaluation
    .groupby("requested_mode")
    .agg(
        source_recall_at_5=("source_hit", "mean"),
        mean_evidence_coverage_at_5=("evidence_coverage", "mean"),
        full_evidence_recall_at_5=("all_evidence", "mean"),
        mrr=("rr", "mean"),
    )
    .sort_values(
        ["full_evidence_recall_at_5", "mrr"],
        ascending=False,
    )
)

display(summary)

direction_summary = (
    auto_eval
    .groupby("direction")
    .agg(
        source_recall_at_5=("source_hit", "mean"),
        mean_evidence_coverage_at_5=("evidence_coverage", "mean"),
        full_evidence_recall_at_5=("all_evidence", "mean"),
        mrr=("rr", "mean"),
        cases=("case", "count"),
    )
    .reset_index()
)

display(direction_summary)

## 5. Inspect the multilingual cases

In [ ]:
for case_name in [
    "rights_transfer_en",
    "rights_general_en",
    "kh_health_object_bg",
]:
    case = next(c for c in CASES if c["name"] == case_name)

    print("\n", "=" * 90)
    print(case_name, "|", case["query"])
    print("=" * 90)

    results = retriever.retrieve(
        case["query"],
        k=8,
        mode="auto",
    )

    display(pd.DataFrame([
        {
            "rank": r["rank"],
            "document": r.get("document"),
            "fund": r.get("fund"),
            "section": r.get("section"),
            "pages": f'{r.get("page_start")}-{r.get("page_end")}',
            "chunk_language": r.get("candidate_language"),
            "effective_mode": r.get("effective_mode"),
            "semantic_score": r.get("semantic_score"),
            "bm25_score": r.get("bm25_score"),
            "text": (r.get("text") or "")[:260],
        }
        for r in results
    ]))

## 6. Diversified top-50 candidate recall

This is a **reranking safety net**, not the primary multilingual fix.
BGE-M3 already achieved strong top-5 multilingual retrieval in the model benchmark, but candidate diversification ensures one source language cannot disappear from a wider reranking pool.

In [ ]:
candidate_rows = []
candidate_cache = {}

for case in CASES:
    candidates = retriever.candidate_set(
        case["query"],
        candidate_k=50,
        global_k=30,
        per_language_k=10,
    )
    candidate_cache[case["name"]] = candidates

    metrics = evaluate_case(candidates, case, k=50)

    candidate_rows.append({
        "case": case["name"],
        "direction": case["direction"],
        "candidate_count": len(candidates),
        "source_hit@50": metrics["source_hit"],
        "evidence_coverage@50": metrics["evidence_coverage"],
        "all_evidence@50": metrics["all_evidence"],
    })

candidate_eval = pd.DataFrame(candidate_rows)
display(candidate_eval)

print(
    "Full evidence recall@50:",
    candidate_eval["all_evidence@50"].mean(),
)

## 7. Optional reranker experiment

The live system does **not** use a reranker. The experiment below is retained to document that decision: BGE reranking improved ordering (MRR) but did not improve top-5 evidence coverage and added substantial latency.

Set `RUN_RERANKER = True` only when you intentionally want to reproduce the benchmark.


In [ ]:
RUN_RERANKER = False

rerank_comparison = pd.DataFrame()

if RUN_RERANKER:
    ## 7. Reranker value vs latency benchmark

    import time

    RERANK_CANDIDATE_K = 10

    reranker = retrieval.MultilingualReranker(
        batch_size=2,
    )

    rows = []

    for case in CASES:
        candidates = candidate_cache[case["name"]][:RERANK_CANDIDATE_K]

        if not candidates:
            continue

        search_query = candidates[0]["search_query"]

        start = time.perf_counter()

        scores = reranker.score(
            search_query,
            [
                retrieval.semantic_text(candidate)
                for candidate in candidates
            ],
        )

        latency = time.perf_counter() - start

        reranked = [
            {
                **candidate,
                "reranker_score": float(score),
            }
            for candidate, score in zip(candidates, scores)
        ]

        reranked.sort(
            key=lambda row: row["reranker_score"],
            reverse=True,
        )

        metrics = evaluate_case(
            reranked,
            case,
            k=5,
        )

        auto_row = auto_eval.loc[
            auto_eval["case"] == case["name"]
        ].iloc[0]

        rows.append({
            "case": case["name"],
            "direction": case["direction"],

            "auto_coverage@5":
                auto_row["evidence_coverage"],

            "rerank_coverage@5":
                metrics["evidence_coverage"],

            "auto_mrr":
                auto_row["rr"],

            "rerank_mrr":
                metrics["rr"],

            "coverage_gain":
                metrics["evidence_coverage"]
                - auto_row["evidence_coverage"],

            "mrr_gain":
                metrics["rr"]
                - auto_row["rr"],

            "reranker_latency_s":
                latency,
        })


    rerank_comparison = pd.DataFrame(rows)

    display(rerank_comparison)
else:
    print("Skipped reranker benchmark. Live retrieval uses mode='auto' without reranking.")


In [ ]:
if not rerank_comparison.empty:
    display(pd.DataFrame([{
        "auto_mean_coverage@5": rerank_comparison["auto_coverage@5"].mean(),
        "rerank_mean_coverage@5": rerank_comparison["rerank_coverage@5"].mean(),
        "auto_mrr": rerank_comparison["auto_mrr"].mean(),
        "rerank_mrr": rerank_comparison["rerank_mrr"].mean(),
        "mean_reranker_latency_s": rerank_comparison["reranker_latency_s"].mean(),
    }]))
else:
    print("Last validated reranker benchmark: coverage@5 0.9444 → 0.9444; MRR 0.8148 → 1.0000; mean latency ≈16.39 s/query.")


## 8. Retrieval conclusion

The controlled model comparison established **BGE-M3** as the production embedding model:

| Case | multilingual-e5-base rank | BGE-M3 rank |
|---|---:|---:|
| EN → BG transfer | 49 | 2 |
| EN → BG rights | 31 | 4 |
| BG → BG transfer | 1 | 1 |
| BG → BG rights | 2 | 4 |
| EN → EN K&H objective | 1 | 1 |
| BG → EN K&H objective | 1 | 1 |

On the nine-case evidence benchmark, production `auto` retrieval achieved **100% source recall@5**, **94.44% mean evidence coverage@5**, **88.89% full-evidence recall@5**, and **0.8148 MRR**. The only partial top-5 case was `enhanced_esg_risk` (0.5 evidence coverage). The diversified top-50 candidate set recovered all required evidence.

The multilingual BGE reranker improved ordering but **did not improve evidence coverage** (0.9444 → 0.9444) and added roughly **16.39 seconds/query**, so it is intentionally excluded from the live path.
